In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

import sys
from pathlib import Path
import random

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from models.directed_gcn_v1 import DirectedGCN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PROCESSED = Path("../data/processed")
MODEL_PATH = Path("../models/directed_gcn_v1.pt")

## Load Dataset

In [2]:
DATA_PROCESSED = Path("../data/processed")

def load_graph(name):
    path = DATA_PROCESSED / f"{name}_final.pt"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    data = torch.load(path, weights_only=False)
    print(f"Loaded dataset: {name}")
    return data.to(device)

datasets = ["elliptic", "paysim"]

## 2. Structural Feature Enrichment

In [3]:
def compute_structural_features(data):
    num_nodes = data.num_nodes
    src, dst = data.edge_index

    in_deg = torch.zeros(num_nodes, device=device)
    in_deg.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

    out_deg = torch.zeros(num_nodes, device=device)
    out_deg.scatter_add_(0, src, torch.ones_like(src, dtype=torch.float))

    total_deg = in_deg + out_deg
    total_deg[total_deg == 0] = 1

    degree_imbalance = torch.abs(in_deg - out_deg) / total_deg
    flow_asymmetry = (out_deg - in_deg) / total_deg

    fraud_labels = data.y.float()

    first_hop = torch.zeros(num_nodes, device=device)
    first_hop.index_add_(0, dst, fraud_labels[src])

    deg_in = torch.zeros(num_nodes, device=device)
    deg_in.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
    deg_in[deg_in == 0] = 1
    first_hop = first_hop / deg_in

    second_hop = torch.zeros(num_nodes, device=device)
    second_hop.index_add_(0, dst, first_hop[src])
    second_hop = second_hop / deg_in

    structural_feats = torch.stack([
        in_deg,
        out_deg,
        degree_imbalance,
        flow_asymmetry,
        second_hop
    ], dim=1)

    return structural_feats

In [4]:
def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob)
    }

In [5]:
def evaluate_model(model, data, device):
    model.eval()
    data = data.to(device)

    with torch.no_grad():
        logits = model(data)

        test_mask = data.test_mask

        preds = logits[test_mask].argmax(dim=1)
        probs = F.softmax(logits[test_mask], dim=1)[:, 1]

    return evaluate_predictions(
        data.y[test_mask].cpu(),
        preds.cpu(),
        probs.cpu()
    )

In [6]:
# Training Function
def train_model(model, data, device, epochs=50, lr=0.001, weight_decay=5e-4):
    model = model.to(device)
    data = data.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(data)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    return model

In [7]:
# Experiment Loop

datasets = ["elliptic", "paysim"]
results = []

for dataset_name in datasets:

    print(f"\nRunning experiment on {dataset_name.upper()}")

    data = load_graph(dataset_name)

    # BASE MODEL (Original Features)

    base_model = DirectedGCN(
        input_dim=data.x.shape[1],
        hidden_dim=64
    ).to(device)

    base_model = train_model(base_model, data, device)

    base_metrics = evaluate_model(base_model, data, device)

    # ENRICHED MODEL (Structural Features)

    structural_features = compute_structural_features(data)

    data_enriched = data.clone()
    data_enriched.x = torch.cat(
        [data.x, structural_features],
        dim=1
    )

    enriched_model = DirectedGCN(
        input_dim=data_enriched.x.shape[1],
        hidden_dim=64
    ).to(device)

    enriched_model = train_model(enriched_model, data_enriched, device)

    enriched_metrics = evaluate_model(enriched_model, data_enriched, device)

    # Store Results
    results.append([
        "DirectedGCN v1 (Base)",
        dataset_name,
        base_metrics["F1"],
        base_metrics["Precision"],
        base_metrics["Recall"],
        base_metrics["AUC"]
    ])

    results.append([
        "DirectedGCN (Enriched)",
        dataset_name,
        enriched_metrics["F1"],
        enriched_metrics["Precision"],
        enriched_metrics["Recall"],
        enriched_metrics["AUC"]
    ])


results_df = pd.DataFrame(
    results,
    columns=["Model", "Dataset", "F1", "Precision", "Recall", "AUC"]
)

print("\nFinal Results:")
print(results_df)


Running experiment on ELLIPTIC


Loaded dataset: elliptic
Epoch 10/50, Loss: 0.2980
Epoch 20/50, Loss: 0.2297
Epoch 30/50, Loss: 0.1978
Epoch 40/50, Loss: 0.1764
Epoch 50/50, Loss: 0.1617
Epoch 10/50, Loss: 0.3197
Epoch 20/50, Loss: 0.2487
Epoch 30/50, Loss: 0.2160
Epoch 40/50, Loss: 0.1914
Epoch 50/50, Loss: 0.1747

Running experiment on PAYSIM
Loaded dataset: paysim
Epoch 10/50, Loss: 0.5914
Epoch 20/50, Loss: 0.4518
Epoch 30/50, Loss: 0.3428
Epoch 40/50, Loss: 0.2591
Epoch 50/50, Loss: 0.1975
Epoch 10/50, Loss: 0.4405
Epoch 20/50, Loss: 0.3127
Epoch 30/50, Loss: 0.2186
Epoch 40/50, Loss: 0.1537
Epoch 50/50, Loss: 0.1137

Final Results:
                    Model   Dataset        F1  Precision    Recall       AUC
0   DirectedGCN v1 (Base)  elliptic  0.318870   0.219368  0.583564  0.816714
1  DirectedGCN (Enriched)  elliptic  0.285240   0.191842  0.555863  0.783145
2   DirectedGCN v1 (Base)    paysim  0.068886   0.606061  0.036519  0.698754
3  DirectedGCN (Enriched)    paysim  0.091165   0.604478  0.049300  0.804781


## 2. Sampling and Direction Sensitivity Study

In [8]:
# Structural Metrics (Graph-level)
def compute_structural_metrics(data):
    src, dst = data.edge_index
    num_nodes = data.num_nodes

    in_deg = torch.zeros(num_nodes, device=device)
    in_deg.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

    out_deg = torch.zeros(num_nodes, device=device)
    out_deg.scatter_add_(0, src, torch.ones_like(src, dtype=torch.float))

    total_deg = in_deg + out_deg
    total_deg[total_deg == 0] = 1

    flow_asymmetry = torch.abs(out_deg - in_deg) / total_deg

    fraud = data.y
    fraud_edges = ((fraud[src] == 1) & (fraud[dst] == 1)).sum().item()
    total_edges = src.shape[0]

    homophily = fraud_edges / total_edges if total_edges > 0 else 0

    return {
        "avg_in_deg": in_deg.mean().item(),
        "avg_out_deg": out_deg.mean().item(),
        "avg_flow_asym": flow_asymmetry.mean().item(),
        "fraud_homophily": homophily
    }



In [9]:
#  Legitimate Node Subsampling
def subsample_graph(data, ratio):
    mask = data.y == 0
    legit_indices = torch.where(mask)[0]

    num_remove = int(len(legit_indices) * ratio)
    remove_nodes = legit_indices[
        torch.randperm(len(legit_indices))[:num_remove]
    ]

    keep_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)
    keep_mask[remove_nodes] = False

    new_data = data.clone()
    new_data.x = data.x[keep_mask]
    new_data.y = data.y[keep_mask]

    idx_map = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    idx_map[keep_mask] = torch.arange(keep_mask.sum(), device=device)

    src, dst = data.edge_index
    edge_mask = keep_mask[src] & keep_mask[dst]

    new_src = idx_map[src[edge_mask]]
    new_dst = idx_map[dst[edge_mask]]

    new_data.edge_index = torch.stack([new_src, new_dst])
    new_data.train_mask = data.train_mask[keep_mask]
    new_data.test_mask = data.test_mask[keep_mask]
    new_data.num_nodes = keep_mask.sum().item()

    return new_data



In [10]:
#  Subsampling + Enrichment Experiment

sampling_ratios = [0.05, 0.10, 0.20]
subsampling_results = []

for dataset_name in datasets:

    print(f"\nRunning Subsampling Study on {dataset_name.upper()}")

    original_data = load_graph(dataset_name)

    for ratio in sampling_ratios:

        print(f"  Sampling {int(ratio*100)}% legitimate nodes")

        sampled_data = subsample_graph(original_data, ratio)

        # Structural Metrics (after sampling)
        structural = compute_structural_metrics(sampled_data)

        #  DirectedGCN v1 (Base Features)
        model_v1 = DirectedGCN(
            input_dim=sampled_data.x.shape[1],
            hidden_dim=64
        ).to(device)

        model_v1 = train_model(model_v1, sampled_data, device)
        metrics_v1 = evaluate_model(model_v1, sampled_data, device)

        #  DirectedGCN v2 (Enriched Features)
        structural_features = compute_structural_features(sampled_data)

        enriched_data = sampled_data.clone()
        enriched_data.x = torch.cat(
            [sampled_data.x, structural_features],
            dim=1
        )

        model_v2 = DirectedGCN(
            input_dim=enriched_data.x.shape[1],
            hidden_dim=64
        ).to(device)

        model_v2 = train_model(model_v2, enriched_data, device)
        metrics_v2 = evaluate_model(model_v2, enriched_data, device)

        # Store Results
        subsampling_results.append([
            dataset_name,
            ratio,
            structural["avg_in_deg"],
            structural["avg_out_deg"],
            structural["avg_flow_asym"],
            structural["fraud_homophily"],
            metrics_v1["F1"],
            metrics_v2["F1"],
            metrics_v1["Recall"],
            metrics_v2["Recall"],
            metrics_v1["AUC"],
            metrics_v2["AUC"]
        ])


Running Subsampling Study on ELLIPTIC
Loaded dataset: elliptic
  Sampling 5% legitimate nodes
Epoch 10/50, Loss: 0.3576
Epoch 20/50, Loss: 0.2691
Epoch 30/50, Loss: 0.2323
Epoch 40/50, Loss: 0.2075
Epoch 50/50, Loss: 0.1900
Epoch 10/50, Loss: 0.3205
Epoch 20/50, Loss: 0.2592
Epoch 30/50, Loss: 0.2272
Epoch 40/50, Loss: 0.2020
Epoch 50/50, Loss: 0.1848
  Sampling 10% legitimate nodes
Epoch 10/50, Loss: 0.3578
Epoch 20/50, Loss: 0.2729
Epoch 30/50, Loss: 0.2384
Epoch 40/50, Loss: 0.2150
Epoch 50/50, Loss: 0.1979
Epoch 10/50, Loss: 0.3024
Epoch 20/50, Loss: 0.2454
Epoch 30/50, Loss: 0.2082
Epoch 40/50, Loss: 0.1845
Epoch 50/50, Loss: 0.1674
  Sampling 20% legitimate nodes
Epoch 10/50, Loss: 0.3544
Epoch 20/50, Loss: 0.2842
Epoch 30/50, Loss: 0.2497
Epoch 40/50, Loss: 0.2249
Epoch 50/50, Loss: 0.2060
Epoch 10/50, Loss: 0.3414
Epoch 20/50, Loss: 0.2861
Epoch 30/50, Loss: 0.2494
Epoch 40/50, Loss: 0.2226
Epoch 50/50, Loss: 0.2033

Running Subsampling Study on PAYSIM
Loaded dataset: paysim
 

In [11]:
subsampling_df = pd.DataFrame(subsampling_results, columns=[
    "Dataset",
    "Sampling_Ratio",
    "Avg_In_Deg",
    "Avg_Out_Deg",
    "Flow_Asym",
    "Fraud_Homophily",
    "DirectedGCN_v1_F1",
    "DirectedGCN_v2_F1",
    "DirectedGCN_v1_Recall",
    "DirectedGCN_v2_Recall",
    "DirectedGCN_v1_AUC",
    "DirectedGCN_v2_AUC"
])

print("\nSubsampling + Structural Enrichment Results:")
print(subsampling_df)


Subsampling + Structural Enrichment Results:
    Dataset  Sampling_Ratio  Avg_In_Deg  Avg_Out_Deg  Flow_Asym  \
0  elliptic            0.05    1.130754     1.130754   0.519510   
1  elliptic            0.10    1.113039     1.113039   0.515135   
2  elliptic            0.20    1.080229     1.080229   0.506636   
3    paysim            0.05    0.523173     0.523173   0.954024   
4    paysim            0.10    0.495798     0.495798   0.907541   
5    paysim            0.20    0.441074     0.441074   0.813389   

   Fraud_Homophily  DirectedGCN_v1_F1  DirectedGCN_v2_F1  \
0         0.004376           0.270026           0.305850   
1         0.004493           0.305906           0.320707   
2         0.004729           0.306042           0.348582   
3         0.000000           0.000000           0.090395   
4         0.000000           0.076353           0.028103   
5         0.000000           0.069284           0.083805   

   DirectedGCN_v1_Recall  DirectedGCN_v2_Recall  DirectedGCN_v1

## 3. Directed GCN Robustness & Noise Testing

### Role of Synthetic Data in Model Evaluation

Synthetic data was used exclusively for stability and noise robustness analysis of the Directed GCN architecture. The purpose of these experiments was to evaluate the behavior of directed message passing under controlled structural perturbations and injected noise. Synthetic graphs allow precise manipulation of directionality, sparsity, and feature corruption levels, enabling systematic stress-testing of the model.

However, synthetic data was not used for model training, hyperparameter tuning, or final performance reporting. Since synthetic datasets do not capture the full complexity and evolving nature of real financial fraud networks, all primary evaluations were conducted on real-world datasets to ensure practical relevance and external validity of the results.

In [12]:
# Corruption / Noise Functions
def remove_random_edges(data, ratio):
    new_data = data.clone()
    edge_index = data.edge_index
    num_edges = edge_index.shape[1]
    num_remove = int(num_edges * ratio)
    if num_remove == 0:
        return new_data
    keep_indices = torch.randperm(num_edges)[num_remove:]
    new_data.edge_index = edge_index[:, keep_indices]
    return new_data

In [13]:
def flip_edge_direction(data, ratio):
    new_data = data.clone()
    edge_index = data.edge_index.clone()
    num_edges = edge_index.shape[1]
    num_flip = int(num_edges * ratio)
    if num_flip == 0:
        return new_data
    flip_indices = torch.randperm(num_edges)[:num_flip]
    src = edge_index[0, flip_indices].clone()
    dst = edge_index[1, flip_indices].clone()
    edge_index[0, flip_indices] = dst
    edge_index[1, flip_indices] = src
    new_data.edge_index = edge_index
    return new_data

In [14]:
def inject_label_noise(data, ratio):
    new_data = data.clone()
    labels = data.y.clone()
    train_indices = torch.where(data.train_mask)[0]
    num_flip = int(len(train_indices) * ratio)
    if num_flip > 0:
        flip_indices = train_indices[torch.randperm(len(train_indices))[:num_flip]]
        labels[flip_indices] = 1 - labels[flip_indices]
    new_data.y = labels
    return new_data

In [15]:
def mask_features(data, ratio):
    new_data = data.clone()
    x = data.x.clone()
    num_features = x.shape[1]
    num_mask = int(num_features * ratio)
    if num_mask == 0:
        return new_data
    mask_indices = torch.randperm(num_features)[:num_mask]
    x[:, mask_indices] = 0
    new_data.x = x
    return new_data

In [16]:
def evaluate_directed_model(model_class, data, enrich=False,
                            hidden_dim=64, lr=0.001, weight_decay=5e-4,
                            epochs=50):
    """Train & evaluate DirectedGCN or enriched DirectedGCN"""
    data = data.to(device)

    if enrich:
        # Append structural features
        structural_feats = compute_structural_features(data)
        data = data.clone()
        data.x = torch.cat([data.x, structural_feats], dim=1)

    input_dim = data.x.shape[1]
    model = model_class(input_dim=input_dim, hidden_dim=hidden_dim).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        logits = model(data)
        test_mask = data.test_mask
        preds = logits[test_mask].argmax(dim=1)
        probs = F.softmax(logits[test_mask], dim=1)[:, 1]
        y_true = data.y[test_mask]

    return {
        "F1": f1_score(y_true.cpu(), preds.cpu(), zero_division=0),
        "Recall": recall_score(y_true.cpu(), preds.cpu(), zero_division=0),
        "AUC": roc_auc_score(y_true.cpu(), probs.cpu()) if len(y_true) > 0 else 0
    }

In [25]:
# DirectedGCN Robustness Study (Synthetic Only: v1 + Enriched)

dataset_name = "synthetic"
corruption_ratios = [0.05, 0.10]

noise_results = []

def prepare_enriched(data):
    """Return a copy of data with structural features appended"""
    data_enriched = data.clone()
    structural_feats = compute_structural_features(data)
    data_enriched.x = torch.cat([data_enriched.x, structural_feats], dim=1)
    return data_enriched


print(f"\nRunning Robustness Study on {dataset_name.upper()}")

data = load_graph(dataset_name)

# Handle Class Imbalance (Prevents 0.0 F1)
train_labels = data.y[data.train_mask]
class_counts = torch.bincount(train_labels).float()
class_weights = 1.0 / (class_counts + 1e-8)
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(device)

print("Class counts:", class_counts.cpu().numpy())
print("Class weights:", class_weights.cpu().numpy())


def train_weighted(model, data, device, epochs=100):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(
            logits[data.train_mask],
            data.y[data.train_mask]
        )
        loss.backward()
        optimizer.step()

    return model


# Train Base Models
print("Training DirectedGCN v1 (Base Features)")
base_model_v1 = DirectedGCN(
    input_dim=data.x.shape[1],
    hidden_dim=64
).to(device)

base_model_v1 = train_weighted(base_model_v1, data, device)


print("Training DirectedGCN Enriched (Structural Features)")
data_enr = prepare_enriched(data)

base_model_enriched = DirectedGCN(
    input_dim=data_enr.x.shape[1],
    hidden_dim=64
).to(device)

base_model_enriched = train_weighted(base_model_enriched, data_enr, device)


# Baseline Evaluation
metrics_v1_base = evaluate_model(base_model_v1, data, device)
metrics_enr_base = evaluate_model(base_model_enriched, data_enr, device)

noise_results.append([
    dataset_name,
    "Baseline",
    0.0,
    metrics_v1_base["F1"],
    metrics_enr_base["F1"]
])


# Corruption Experiments
print("\nStarting corruption experiments")
print("Corruption ratios:", corruption_ratios)

for ratio in corruption_ratios:
    print(f"\nProcessing corruption ratio: {ratio}")

    try:
        # Edge Removal
        removed = remove_random_edges(data, ratio)
        removed_enr = prepare_enriched(removed)

        metrics_v1_r = evaluate_model(base_model_v1, removed, device)
        metrics_enr_r = evaluate_model(base_model_enriched, removed_enr, device)

        noise_results.append([
            dataset_name,
            "Edge Removal",
            ratio,
            metrics_v1_r["F1"],
            metrics_enr_r["F1"]
        ])

        # Direction Flip
        flipped = flip_edge_direction(data, ratio)
        flipped_enr = prepare_enriched(flipped)

        metrics_v1_f = evaluate_model(base_model_v1, flipped, device)
        metrics_enr_f = evaluate_model(base_model_enriched, flipped_enr, device)

        noise_results.append([
            dataset_name,
            "Direction Flip",
            ratio,
            metrics_v1_f["F1"],
            metrics_enr_f["F1"]
        ])

        # Label Noise
        noisy = inject_label_noise(data, ratio)
        noisy_enr = prepare_enriched(noisy)

        metrics_v1_l = evaluate_model(base_model_v1, noisy, device)
        metrics_enr_l = evaluate_model(base_model_enriched, noisy_enr, device)

        noise_results.append([
            dataset_name,
            "Label Noise",
            ratio,
            metrics_v1_l["F1"],
            metrics_enr_l["F1"]
        ])

        # Feature Masking
        masked = mask_features(data, ratio)
        masked_enr = prepare_enriched(masked)

        metrics_v1_m = evaluate_model(base_model_v1, masked, device)
        metrics_enr_m = evaluate_model(base_model_enriched, masked_enr, device)

        noise_results.append([
            dataset_name,
            "Feature Mask",
            ratio,
            metrics_v1_m["F1"],
            metrics_enr_m["F1"]
        ])

    except Exception as e:
        print(f"Error at ratio {ratio}: {e}")
        continue



Running Robustness Study on SYNTHETIC
Loaded dataset: synthetic
Class counts: [3065.  335.]
Class weights: [0.09852941 0.9014706 ]
Training DirectedGCN v1 (Base Features)
Training DirectedGCN Enriched (Structural Features)

Starting corruption experiments
Corruption ratios: [0.05, 0.1]

Processing corruption ratio: 0.05

Processing corruption ratio: 0.1


In [26]:
# Compile results
noise_df = pd.DataFrame(noise_results, columns=[
    "Dataset", "Corruption_Type", "Ratio", "DirectedGCN_v1_F1", "DirectedGCN_Enriched_F1"
])
print("\nDirectedGCN Robustness Results (v1 vs Enriched):")
print(noise_df)


DirectedGCN Robustness Results (v1 vs Enriched):
     Dataset Corruption_Type  Ratio  DirectedGCN_v1_F1  \
0  synthetic        Baseline   0.00           0.837607   
1  synthetic    Edge Removal   0.05           0.798354   
2  synthetic  Direction Flip   0.05           0.733333   
3  synthetic     Label Noise   0.05           0.837607   
4  synthetic    Feature Mask   0.05           0.837607   
5  synthetic    Edge Removal   0.10           0.820084   
6  synthetic  Direction Flip   0.10           0.620690   
7  synthetic     Label Noise   0.10           0.837607   
8  synthetic    Feature Mask   0.10           0.820084   

   DirectedGCN_Enriched_F1  
0                 0.844828  
1                 0.855895  
2                 0.863436  
3                 0.844828  
4                 0.844828  
5                 0.848485  
6                 0.899083  
7                 0.848485  
8                 0.848485  


## 4. Hyper Parameter Tuning

In [27]:

import itertools
import copy
from tqdm import tqdm
from itertools import product
import pandas as pd
import torch

In [28]:
# Define hyperparameter search space

param_grid = {
    "hidden_dim": [32, 64, 128],
    "lr": [0.01, 0.001, 0.0005],
    "weight_decay": [0, 5e-4, 1e-3],
    "dropout": [0.0, 0.2, 0.5]
}

# Generate all possible combinations
param_combinations = list(itertools.product(
    param_grid["hidden_dim"],
    param_grid["lr"],
    param_grid["weight_decay"],
    param_grid["dropout"]
))


param_combinations = param_combinations[:15]  



In [29]:
# Function to train and evaluate v3
def train_evaluate_v3(data, hidden_dim, lr, weight_decay, dropout, epochs=50):
    data = data.to(device)

    # Enrich features (v2 style)
    structural_feats = compute_structural_features(data)
    data_enriched = data.clone()
    data_enriched.x = torch.cat([data.x, structural_feats], dim=1)

    # Initialize model with dropout
    model = DirectedGCN(
        input_dim=data_enriched.x.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(data_enriched)
        loss = criterion(logits[data_enriched.train_mask], data_enriched.y[data_enriched.train_mask])
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        logits = model(data_enriched)
        test_mask = data_enriched.test_mask
        preds = logits[test_mask].argmax(dim=1)
        probs = F.softmax(logits[test_mask], dim=1)[:, 1]
        y_true = data_enriched.y[test_mask]

    metrics = {
        "F1": f1_score(y_true.cpu(), preds.cpu(), zero_division=0),
        "Recall": recall_score(y_true.cpu(), preds.cpu(), zero_division=0),
        "AUC": roc_auc_score(y_true.cpu(), probs.cpu()) if len(y_true) > 0 else 0
    }

    return metrics, copy.deepcopy(model)


In [30]:
# Hyperparameter tuning

datasets = ["elliptic", "paysim"]

hidden_dims = [32, 64, 128]
lrs = [0.01, 0.005, 0.001]
weight_decays = [0, 1e-4, 5e-4]

param_combinations = list(product(hidden_dims, lrs, weight_decays))

tuning_results = []
best_models = {}

def train_evaluate_v3_fast(data_enriched, hidden_dim, lr, weight_decay,
                           epochs=25, patience=5):
    """
    Faster training with early stopping
    """
    model = DirectedGCN(
        input_dim=data_enriched.x.shape[1],
        hidden_dim=hidden_dim
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.CrossEntropyLoss()

    best_loss = float("inf")
    patience_counter = 0

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(data_enriched)
        loss = criterion(
            logits[data_enriched.train_mask],
            data_enriched.y[data_enriched.train_mask]
        )
        loss.backward()
        optimizer.step()

        # Early stopping check
        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Evaluation
    model.eval()
    with torch.no_grad():
        logits = model(data_enriched)
        test_mask = data_enriched.test_mask
        preds = logits[test_mask].argmax(dim=1)
        probs = torch.softmax(logits[test_mask], dim=1)[:, 1]
        y_true = data_enriched.y[test_mask]

    metrics = {
        "F1": f1_score(y_true.cpu(), preds.cpu(), zero_division=0),
        "Recall": recall_score(y_true.cpu(), preds.cpu(), zero_division=0),
        "AUC": roc_auc_score(y_true.cpu(), probs.cpu()) if len(y_true) > 0 else 0
    }

    return metrics, model


# Run tuning
for dataset_name in datasets:

    print(f"\n Tuning on {dataset_name.upper()}")

    data = load_graph(dataset_name).to(device)

    #  Compute structural features ONCE
    structural_feats = compute_structural_features(data)
    data_enriched = data.clone()
    data_enriched.x = torch.cat([data.x, structural_feats], dim=1)

    best_f1 = -1
    best_params = None
    best_model = None

    for hidden_dim, lr, wd in tqdm(param_combinations, desc=f"{dataset_name} tuning"):
        metrics, model = train_evaluate_v3_fast(
            data_enriched, hidden_dim, lr, wd
        )

        tuning_results.append([
            dataset_name, hidden_dim, lr, wd,
            metrics["F1"], metrics["Recall"], metrics["AUC"]
        ])

        if metrics["F1"] > best_f1:
            best_f1 = metrics["F1"]
            best_params = (hidden_dim, lr, wd)
            best_model = model

    best_models[dataset_name] = {
        "params": best_params,
        "model": best_model
    }

    print(
        f" Best params: Hidden={best_params[0]}, "
        f"LR={best_params[1]}, WD={best_params[2]}, "
        f"F1={best_f1:.4f}"
    )


# Save results
tuning_df = pd.DataFrame(
    tuning_results,
    columns=["Dataset", "Hidden_Dim", "LR", "Weight_Decay", "F1", "Recall", "AUC"]
)

print("\nTop Results:")
print(tuning_df.sort_values("F1", ascending=False).head(10))


 Tuning on ELLIPTIC
Loaded dataset: elliptic


elliptic tuning: 100%|█████████████████████████████████████████████████████████████████| 27/27 [16:24<00:00, 36.48s/it]


 Best params: Hidden=64, LR=0.01, WD=0, F1=0.4544

 Tuning on PAYSIM
Loaded dataset: paysim


paysim tuning: 100%|███████████████████████████████████████████████████████████████████| 27/27 [09:10<00:00, 20.40s/it]

 Best params: Hidden=128, LR=0.01, WD=0, F1=0.2142

Top Results:
     Dataset  Hidden_Dim     LR  Weight_Decay        F1    Recall       AUC
9   elliptic          64  0.010        0.0000  0.454434  0.513389  0.847797
19  elliptic         128  0.010        0.0001  0.377785  0.508772  0.825750
22  elliptic         128  0.005        0.0001  0.375195  0.555863  0.828537
21  elliptic         128  0.005        0.0000  0.374728  0.555863  0.819808
23  elliptic         128  0.005        0.0005  0.369310  0.595568  0.830577
25  elliptic         128  0.001        0.0001  0.349251  0.344414  0.786861
20  elliptic         128  0.010        0.0005  0.345029  0.490305  0.819518
18  elliptic         128  0.010        0.0000  0.344344  0.476454  0.826396
10  elliptic          64  0.010        0.0001  0.340117  0.563250  0.816750
11  elliptic          64  0.010        0.0005  0.339317  0.445060  0.815218


## Final Model Ranking 

In [32]:
# FINAL MODEL PERFORMANCE SUMMARY

final_summary = []

# Base vs Enriched Results
for _, row in results_df.iterrows():
    final_summary.append([
        row["Dataset"],
        row["Model"],
        "Standard Training",
        row["F1"],
        row["Recall"],
        row["AUC"]
    ])

#  Best Tuned Model (v3)
for dataset_name in best_models.keys():
    best_model = best_models[dataset_name]["model"]
    data = load_graph(dataset_name)

    # Enrich features
    structural_feats = compute_structural_features(data)
    data_enriched = data.clone()
    data_enriched.x = torch.cat([data.x, structural_feats], dim=1)

    metrics = evaluate_model(best_model, data_enriched, device)

    final_summary.append([
        dataset_name,
        "DirectedGCN v3 (Tuned)",
        "Hyperparameter Tuned",
        metrics["F1"],
        metrics["Recall"],
        metrics["AUC"]
    ])

# Create DataFrame
final_comparison_df = pd.DataFrame(
    final_summary,
    columns=["Dataset", "Model", "Setting", "F1", "Recall", "AUC"]
)

print("\n Final Model Ranking (Sorted by F1)")
print(final_comparison_df.sort_values(["Dataset", "F1"], ascending=[True, False]))

Loaded dataset: elliptic
Loaded dataset: paysim

 Final Model Ranking (Sorted by F1)
    Dataset                   Model               Setting        F1    Recall  \
4  elliptic  DirectedGCN v3 (Tuned)  Hyperparameter Tuned  0.454434  0.513389   
0  elliptic   DirectedGCN v1 (Base)     Standard Training  0.318870  0.583564   
1  elliptic  DirectedGCN (Enriched)     Standard Training  0.285240  0.555863   
5    paysim  DirectedGCN v3 (Tuned)  Hyperparameter Tuned  0.214151  0.138162   
3    paysim  DirectedGCN (Enriched)     Standard Training  0.091165  0.049300   
2    paysim   DirectedGCN v1 (Base)     Standard Training  0.068886  0.036519   

        AUC  
4  0.847797  
0  0.816714  
1  0.783145  
5  0.881315  
3  0.804781  
2  0.698754  


In [31]:
# SELECT GLOBAL BEST MODEL

best_global = final_comparison_df.sort_values("F1", ascending=False).iloc[0]

best_dataset = best_global["Dataset"]
best_model_name = best_global["Model"]

print(f"\n Best Overall Model:")
print(f"Dataset: {best_dataset}")
print(f"Model: {best_model_name}")
print(f"F1 Score: {best_global['F1']:.4f}")


 Best Overall Model:
Dataset: elliptic
Model: DirectedGCN v3 (Tuned)
F1 Score: 0.4286


In [32]:
# SAVE BEST MODEL

save_path = Path("../models")
save_path.mkdir(exist_ok=True)

best_model_object = best_models[best_dataset]["model"]
best_params = best_models[best_dataset]["params"]

# Get enriched input dimension
data = load_graph(best_dataset)
structural_feats = compute_structural_features(data)
data_enriched = data.clone()
data_enriched.x = torch.cat([data.x, structural_feats], dim=1)

model_save_dict = {
    "dataset": best_dataset,
    "model_name": best_model_name,
    "state_dict": best_model_object.state_dict(),
    "hidden_dim": best_params[0],
    "learning_rate": best_params[1],
    "weight_decay": best_params[2],
    "input_dim": data_enriched.x.shape[1]
}

model_file = save_path / "directed_gcn_best.pt"
torch.save(model_save_dict, model_file)

print(f"\n Best model saved at: {model_file}")

Loaded dataset: elliptic

 Best model saved at: ..\models\directed_gcn_best.pt


In [34]:
from pathlib import Path

# Create results directory if it doesn't exist
results_path = Path("../results")
results_path.mkdir(parents=True, exist_ok=True)

# Save all result tables
final_comparison_df.to_csv(results_path / "final_model_comparison.csv", index=False)
tuning_df.to_csv(results_path / "hyperparameter_tuning_results.csv", index=False)
subsampling_df.to_csv(results_path / "subsampling_results.csv", index=False)
noise_df.to_csv(results_path / "robustness_results.csv", index=False)

print("\nAll results saved successfully to ../results/")


All results saved successfully to ../results/
